In [1]:
# !pip uninstall -y protobuf
# !pip install protobuf==3.20.3

Look at:
event type
meta alarm
meta state
meta value
meta battery perc etc

## Decoding

In [1]:
import pandas as pd
events = pd.read_csv('/Users/aishanipradhan/Desktop/capstone/unzipped_contents/Data/customer1-events-1-2025-to-3-2025.csv')

/var/folders/_5/zwklnk315yqbrfhydp3_pj5w0000gn/T/ipykernel_60358/2152916637.py:2: DtypeWarning: Columns (7,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  events = pd.read_csv('/Users/aishanipradhan/Desktop/capstone/unzipped_contents/Data/customer1-events-1-2025-to-3-2025.csv')


In [2]:
# Import your protobuf classes
from ctl.iox import (
    location_pb2,
    readings_pb2,
    alarm_pb2,
    pf_instrument_config_pb2,
    cal_record_pb2,
    inst_mode_pb2,
    inst_debug_pb2,
    inst_cloud_response_pb2,
    cico_pb2,
    battery_pb2,
    sensor_states_record_pb2,
    tag_pb2,
    connectivity_pb2
)

In [3]:
EVENT_TYPE_TO_PROTO = {
    # Alarm-related
    "ALARM": alarm_pb2.AlarmRecord,
    "WARNING": alarm_pb2.AlarmRecord,
    "NOTIFICATION": alarm_pb2.AlarmRecord,

    # Battery
    "BATTERY": battery_pb2.BatteryRecord,
    "BATTERY_INFO": tag_pb2.BatteryPayload,

    # Location
    "LOCATION": location_pb2.LocationRecord,

    # Calibration / config
    "CALIBRATION": cal_record_pb2.CalRecord,

    # Cloud / connectivity
    "CLOUD_RESPONSE": inst_cloud_response_pb2.CloudResponse,
    "CONNECTIVITY": connectivity_pb2.ConnectivityMsg,  # confirm exists

    # Tag / CICO
    "CICO": cico_pb2.TagRecord,

    # Instrument state
    "MODE": inst_mode_pb2.InstrumentModeRecord,

    # Sensor readings (if present elsewhere)
    "READINGS": readings_pb2.ReadingsRecord,

    # Legacy / unsupported
    "GENERIC": None,        # legacy 5-star link
    "GRID_ACTION": None,    # no pb2 provided
}


In [4]:
import base64
import math
import pandas as pd

def decode_by_event_type(encoded_proto, event_type):
    if pd.isna(encoded_proto) or not event_type:
        return None

    proto_cls = EVENT_TYPE_TO_PROTO.get(event_type)

    # Explicitly skip unsupported / legacy events
    if proto_cls is None:
        return None

    try:
        # Decode base64
        if isinstance(encoded_proto, (bytes, bytearray)):
            raw = encoded_proto
        else:
            raw = base64.b64decode(encoded_proto)

        # Parse protobuf
        msg = proto_cls()
        msg.ParseFromString(raw)
        return msg

    except Exception as e:
        # Fail loudly but safely
        print(f"Failed to decode event_type={event_type}: {e}")
        return None

events["decoded_message"] = events.apply(
    lambda row: decode_by_event_type(
        row["META_ENCODED_PROTO"],
        row["EVENT_TYPE"]
    ),
    axis=1
)

events["decoded_type"] = events["decoded_message"].apply(
    lambda x: x.DESCRIPTOR.full_name if x else None
)


In [12]:
events["SESSION_ID"].nunique()

22334

In [11]:
events[events["EVENT_TYPE"] == "LOCATION"]["SESSION_ID"].nunique()

13759

In [7]:
# events_sample = (
#     events
#     .groupby("EVENT_TYPE", group_keys=False)
#     .head(10)
# )

# events_sample.count()

In [8]:
#events_sample.to_csv('/Users/aishanipradhan/Desktop/capstone/decoded_events_sample.csv', index=False)

In [9]:
#events[events["SESSION_ID"] == "891b82e2-c2d2-4c9c-8b13-95761fe49cb9"].to_csv('/Users/aishanipradhan/Desktop/capstone/session_891b82e2.csv', index=False)

## EDA

Check sessions with more than one recorded date.

In [11]:
# filter first (critical for memory)
grid_events = events.loc[
    events["EVENT_TYPE"] != "GRID_ACTION",
    ["SESSION_ID", "LOGGED_AT_DATE"]
].dropna()

bad_sessions = (
    grid_events
    .groupby("SESSION_ID")["LOGGED_AT_DATE"]
    .nunique()
    .loc[lambda x: x > 1]
)

print(bad_sessions.index.tolist())


['0013bc07-8684-4377-bcb3-462110f3df43', '00264571-28b7-4f9e-bdf5-3976bba57b5f', '002aea5f-0a92-4435-9a3e-de861b9c944f', '00579ec5-016a-403b-a0de-985dbb6e9175', '006d6855-af77-441e-8a93-ee086820f36b', '0099da5d-ff8c-45bc-863b-bab17976cae0', '00c35602-5518-48a5-8ef9-ba6b6b8dcee4', '00c91d31-2233-4a49-9758-d91f27f3bad8', '00cef0b3-c51e-444f-a5e0-2958fd297d01', '00d040a2-9685-44ca-8c01-af40a3fa42e0', '00da94f0-6114-422f-b073-d4e9085cf9ed', '00dbd34b-0647-4799-932d-09460b39dfc1', '0107c6d8-5610-4d97-a5d4-34403cf4c349', '0113737c-59e9-4f4a-9c1e-014540748764', '01281da7-b018-4bb0-9871-827ce5c008c3', '012c3027-760e-47fb-8b3d-f02a8d0bbe6b', '012fb688-6399-4eec-bfa5-e75757e86bb1', '01720915-8a4f-4f42-a45f-7abb84e7d7ed', '01931b6e-61f4-4c82-8bce-37b7bc2cc971', '02339d47-bd2e-4245-afcd-f8a1dd124d9e', '025b3dbc-3df6-48cd-ba02-46b2a6b61e0a', '02609c32-b443-4d10-bda1-168175a31c8d', '026923b0-3130-426f-9ba1-0e136a9b6f31', '02695c92-3d40-492c-b181-3b14c35544e7', '026bbab3-bd0d-42a3-9bc9-ffb9971849e4',

In [15]:

df = (
    events.loc[
        events["SESSION_ID"].isin(bad_sessions.index),
        ["SESSION_ID", "LOGGED_AT_DATE", "LOGGED_AT", "decoded_message"]
    ]
    .dropna(subset=["LOGGED_AT_DATE"])
)

df = df.sort_values(["SESSION_ID", "LOGGED_AT_DATE"])

date_change = (
    df.groupby("SESSION_ID")["LOGGED_AT_DATE"]
      .apply(lambda x: x.ne(x.shift()))
)

rows = []

rows = []

for sid, grp in df.groupby("SESSION_ID", sort=False):
    grp = grp.sort_values("LOGGED_AT_DATE")

    change_pos = grp["LOGGED_AT_DATE"].ne(
        grp["LOGGED_AT_DATE"].shift()
    ).to_numpy().nonzero()[0][1:]  # skip first row

    for pos in change_pos:
        rows.append(grp.iloc[pos - 1])
        rows.append(grp.iloc[pos])

result = (
    pd.DataFrame(rows)
    .sort_values(["SESSION_ID", "LOGGED_AT"])
    .reset_index(drop=True)
)




                                SESSION_ID LOGGED_AT_DATE  \
0     0013bc07-8684-4377-bcb3-462110f3df43     2025-03-19   
1     0013bc07-8684-4377-bcb3-462110f3df43     2025-03-20   
2     00264571-28b7-4f9e-bdf5-3976bba57b5f     2025-03-03   
3     00264571-28b7-4f9e-bdf5-3976bba57b5f     2025-03-04   
4     002aea5f-0a92-4435-9a3e-de861b9c944f     2025-03-27   
...                                    ...            ...   
6803  ffd47428-3754-4c7a-8dab-9a93a3e51169     2025-01-31   
6804  ffddaf0d-a75b-4a33-bc0c-b9a9345afda1     2025-02-21   
6805  ffddaf0d-a75b-4a33-bc0c-b9a9345afda1     2025-02-22   
6806  fffa1bdf-1252-4c7a-ba5f-33ec87f12402     2025-03-05   
6807  fffa1bdf-1252-4c7a-ba5f-33ec87f12402     2025-03-06   

                    LOGGED_AT  \
0     2025-03-19 18:09:34.000   
1     2025-03-20 02:54:39.000   
2     2025-03-03 21:31:33.000   
3     2025-03-04 00:21:32.000   
4     2025-03-27 21:43:58.000   
...                       ...   
6803  2025-01-31 00:49:14.000   
680

In [17]:
display(result)

,SESSION_ID,LOGGED_AT_DATE,LOGGED_AT,decoded_message
0,0013bc07-8684-4377-bcb3-462110f3df43,2025-03-19,2025-03-19 18:09:34.000,longitude: 255169193\nlatitude: 603151133\nalt...
1,0013bc07-8684-4377-bcb3-462110f3df43,2025-03-20,2025-03-20 02:54:39.000,longitude: 255223406\nlatitude: 603156826\nalt...
2,00264571-28b7-4f9e-bdf5-3976bba57b5f,2025-03-03,2025-03-03 21:31:33.000,longitude: 255456128\nlatitude: 603063321\nalt...
3,00264571-28b7-4f9e-bdf5-3976bba57b5f,2025-03-04,2025-03-04 00:21:32.000,longitude: 255457536\nlatitude: 603058470\nalt...
4,002aea5f-0a92-4435-9a3e-de861b9c944f,2025-03-27,2025-03-27 21:43:58.000,longitude: 255283735\nlatitude: 603147490\nalt...
...,...,...,...,...
6803,ffd47428-3754-4c7a-8dab-9a93a3e51169,2025-01-31,2025-01-31 00:49:14.000,longitude: 255201896\nlatitude: 603103983\nalt...
6804,ffddaf0d-a75b-4a33-bc0c-b9a9345afda1,2025-02-21,2025-02-21 21:51:30.000,longitude: 255184521\nlatitude: 603139481\nalt...
6805,ffddaf0d-a75b-4a33-bc0c-b9a9345afda1,2025-02-22,2025-02-22 03:20:09.000,longitude: 255166535\nlatitude: 603154251\nalt...
6806,fffa1bdf-1252-4c7a-ba5f-33ec87f12402,2025-03-05,2025-03-05 23:48:41.000,longitude: 255074686\nlatitude: 603104990\nalt...


### Time Gap Analysis for Location

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime

# Filter LOCATION events
print(datetime.now(), "Filtering LOCATION events...")
loc = events.loc[
    events["EVENT_TYPE"] == "LOCATION",
    ["SESSION_ID", "LOGGED_AT", "decoded_message"]
].dropna(subset=["LOGGED_AT", "decoded_message"])

loc["LOGGED_AT"] = pd.to_datetime(loc["LOGGED_AT"])

# Latitude and longitude extraction using regex
import pandas as pd

def extract_lat_lon(msg):
    lat = lon = None
    if isinstance(msg, str):
        for line in msg.splitlines():  # handles \n and \r\n
            if line.strip().startswith("latitude:"):
                try:
                    lat = int(line.split(":")[1].strip())
                except:
                    lat = None
            elif line.strip().startswith("longitude:"):
                try:
                    lon = int(line.split(":")[1].strip())
                except:
                    lon = None
    return pd.Series([lat, lon])

# Ensure decoded_message is string
loc["decoded_message"] = loc["decoded_message"].astype(str)
print(datetime.now(), "Starting lat/lon extraction...")
# Apply extraction
loc[["latitude", "longitude"]] = loc["decoded_message"].apply(extract_lat_lon)

# Scale last 7 digits to decimals
loc["latitude"]  = loc["latitude"].astype(float) / 1e7
loc["longitude"] = loc["longitude"].astype(float) / 1e7

print(datetime.now(), "Completed lat/lon extraction.")

# Flag valid GPS coordinates
loc["gps_valid"] = loc["latitude"].between(-90, 90) & loc["longitude"].between(-180, 180)

# Sort by session and time
loc = loc.sort_values(["SESSION_ID", "LOGGED_AT"])

# Compute previous row info per session
loc["prev_time"] = loc.groupby("SESSION_ID")["LOGGED_AT"].shift()
loc["prev_lat"]  = loc.groupby("SESSION_ID")["latitude"].shift()
loc["prev_lon"]  = loc.groupby("SESSION_ID")["longitude"].shift()
loc["prev_decoded_message"] = loc.groupby("SESSION_ID")["decoded_message"].shift()

print(datetime.now(), "starting distance and time difference calculations...")
# Haversine distance (vectorized)
R = 6371000  # Earth radius in meters

lat1 = np.radians(loc["prev_lat"])
lon1 = np.radians(loc["prev_lon"])
lat2 = np.radians(loc["latitude"])
lon2 = np.radians(loc["longitude"])

dlat = lat2 - lat1
dlon = lon2 - lon1

a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
loc["distance_m"] = 2 * R * np.arcsin(np.sqrt(a))

# Time difference
loc["time_diff_hours"] = (loc["LOGGED_AT"] - loc["prev_time"]).dt.total_seconds() / 3600

# Max time gap per session
idx = loc.groupby("SESSION_ID")["time_diff_hours"].idxmax().dropna()

# Assemble result dataframe
result = loc.loc[idx, [
    "SESSION_ID",
    "prev_time", "LOGGED_AT",
    "prev_lat", "prev_lon",
    "latitude", "longitude",
    "time_diff_hours",
    "distance_m",
    "prev_decoded_message",
    "decoded_message",
    "gps_valid"
]].rename(columns={
    "prev_time": "start_time",
    "LOGGED_AT": "end_time",
    "prev_lat": "start_lat",
    "prev_lon": "start_lon",
    "latitude": "end_lat",
    "longitude": "end_lon",
    "prev_decoded_message": "start_decoded_message",
    "decoded_message": "end_decoded_message"
}).sort_values("time_diff_hours", ascending=False).reset_index(drop=True)


2026-02-11 00:27:52.904396 Filtering LOCATION events...
2026-02-11 00:28:04.972620 Starting lat/lon extraction...
2026-02-11 00:39:55.146477 Completed lat/lon extraction.
2026-02-11 00:39:56.593918 starting distance and time difference calculations...
                             SESSION_ID          start_time  \
0  5afdee58-91a4-4361-8a2d-7e15c20c40b6 2025-03-12 18:03:55   
1  c5516f0d-a69a-4c9b-a79c-bbe9a2da876b 2025-02-16 13:21:01   
2  662878c8-17ba-4094-9b99-a9fb7dfbb398 2025-01-08 05:54:00   
3  69eb7d4c-b716-4778-9215-f4a4e959faa1 2025-03-13 05:42:06   
4  4bf094fc-c9c7-4fcc-9020-132ade47758b 2025-03-05 06:26:42   

             end_time   start_lat   start_lon     end_lat     end_lon  \
0 2025-03-15 04:59:43   60.306580   25.518663  214.748365  214.748365   
1 2025-02-17 05:00:48   60.306636   25.518713   60.306668   25.518714   
2 2025-01-08 19:37:32  214.748365  214.748365   60.309943   25.525554   
3 2025-03-13 18:53:04  214.748365  214.748365   60.308483   25.543355   
4 20

In [9]:
result.to_csv('/Users/aishanipradhan/Desktop/capstone/longest_location_gaps.csv', index=False)

In [7]:
result

,SESSION_ID,start_time,end_time,start_lat,start_lon,end_lat,end_lon,time_diff_hours,distance_m,start_decoded_message,end_decoded_message
0,5afdee58-91a4-4361-8a2d-7e15c20c40b6,2025-03-12 18:03:55,2025-03-15 04:59:43,60.306580,25.518663,214.748365,214.748365,58.930000,1.060334e+07,longitude: 255186626\nlatitude: 603065796\nalt...,longitude: 2147483647\nlatitude: 2147483647\ne...
1,c5516f0d-a69a-4c9b-a79c-bbe9a2da876b,2025-02-16 13:21:01,2025-02-17 05:00:48,60.306636,25.518713,60.306668,25.518714,15.663056,3.547546e+00,longitude: 255187133\nlatitude: 603066361\nalt...,longitude: 255187143\nlatitude: 603066680\nalt...
2,662878c8-17ba-4094-9b99-a9fb7dfbb398,2025-01-08 05:54:00,2025-01-08 19:37:32,214.748365,214.748365,60.309943,25.525554,13.725556,1.060366e+07,longitude: 2147483647\nlatitude: 2147483647\ne...,longitude: 255255541\nlatitude: 603099433\nalt...
3,69eb7d4c-b716-4778-9215-f4a4e959faa1,2025-03-13 05:42:06,2025-03-13 18:53:04,214.748365,214.748365,60.308483,25.543355,13.182778,1.060337e+07,longitude: 2147483647\nlatitude: 2147483647\n,longitude: 255433553\nlatitude: 603084828\nalt...
4,4bf094fc-c9c7-4fcc-9020-132ade47758b,2025-03-05 06:26:42,2025-03-05 19:36:41,214.748365,214.748365,60.311796,25.525685,13.166389,1.060386e+07,longitude: 2147483647\nlatitude: 2147483647\n,longitude: 255256853\nlatitude: 603117960\nalt...
...,...,...,...,...,...,...,...,...,...,...,...
11845,1afa3ce3-f1cd-4ff4-81b6-6061bbb89d18,2025-02-26 04:57:15,2025-02-26 04:57:25,214.748365,214.748365,60.306608,25.518804,0.002778,1.060334e+07,longitude: 2147483647\nlatitude: 2147483647\n,longitude: 255188043\nlatitude: 603066078\nalt...
11846,075e2abc-5ed0-4078-9f68-e9b0e29a8f3a,2025-02-18 05:01:03,2025-02-18 05:01:12,214.748365,214.748365,60.306640,25.518750,0.002500,1.060335e+07,longitude: 2147483647\nlatitude: 2147483647\n,longitude: 255187500\nlatitude: 603066395\nalt...
11847,b0c6375e-fe35-4450-b06b-880df2357e42,2025-03-02 05:00:35,2025-03-02 05:00:44,214.748365,214.748365,60.306576,25.518627,0.002500,1.060334e+07,longitude: 2147483647\nlatitude: 2147483647\n,longitude: 255186266\nlatitude: 603065763\nalt...
11848,3b38a3fd-1f54-40d6-8395-1bdc8eff811f,2025-03-04 09:46:32,2025-03-04 09:46:37,60.305020,25.503316,60.304782,25.503225,0.001389,2.693596e+01,longitude: 255033156\nlatitude: 603050198\nalt...,longitude: 255032245\nlatitude: 603047818\nalt...
